<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #1: growing vs. declining pages**

The paper finds that growing pages are longer (3.2K vs 2.3K words) and recommends expanding short pages to help them grow.

**Label:** a page is "growing" or "declining" based only on its impressions over the last 30 days vs. the previous 30 days.

**Validation:** The comparison of group averages clearly shows that length and growth are associated. One question is whether length itself drives growth, since content type may play a role: some types (guides, comparisons) are naturally long and also earn more impressions. A useful next step would be to compare long and short pages within the same content type.

<br>

<br>


**Finding #3: CTR by position tier**

The logic holds: CTR falls as position worsens, so the recommended actions fit each tier. Pages already on page 1 gain most from better titles and descriptions (snippets), while pages lower down need stronger content to climb back to page 1.

**Label:** the outcome measured is weighted CTR, calculated as total clicks divided by total impressions within each position tier (Top 3, Page 1, Striking, Page 3-5, Deep).

**Validation:**  CTR does decline as position worsens, so it makes sense to strengthen content on lower-ranked pages and refine snippets on page 1. One observation is that the chart shows a small gap between page 1 and page 2 (0.339% vs 0.325%), while the sharp drop comes between page 2 and pages 3-5 (0.325% vs 0.163%). So the data best supports keeping pages from falling past page 2.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Load data: features and "is_declining" label

In [ ]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login


TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY   = f"{BASE}/fact_content_daily_performance/**/*.parquet"
SRC_CLIENTS = f"{BASE}/dim_clients.parquet"
SRC_CONTENT = f"{BASE}/dim_content.parquet"


FEAT_START, FEAT_END   = "2026-01-01", "2026-02-28"
MID                    = "2026-01-31"
LABEL_START, LABEL_END = "2026-03-01", "2026-04-30"
MIN_IMPR = 100


con.execute(f"""
CREATE OR REPLACE TEMP TABLE content_slice AS
SELECT d.client_hash_id, d.content_hash_id
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN read_parquet('{SRC_CLIENTS}') c USING (client_hash_id)
WHERE d.month IN ('2026-01','2026-02')
  AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
  AND CAST(c.gsc_data_start AS DATE) < DATE '{FEAT_START}'
GROUP BY d.client_hash_id, d.content_hash_id
HAVING SUM(d.gsc_impressions) >= {MIN_IMPR}
""")


con.execute(f"""
CREATE OR REPLACE TEMP TABLE dataset AS
WITH daily AS (
    SELECT d.client_hash_id, d.content_hash_id, d.report_date,
           d.gsc_impressions, d.gsc_clicks, d.gsc_sum_position,
           d.ga4_pageviews, d.ga4_sessions, d.ga4_users,
           d.ga4_engaged_sessions, d.ga4_total_engagement_sec,
           COALESCE(d.report_date >= CAST(c.ga4_data_start AS DATE), FALSE) AS ga4_ok
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN content_slice s
      ON s.client_hash_id = d.client_hash_id AND s.content_hash_id = d.content_hash_id
    JOIN read_parquet('{SRC_CLIENTS}') c ON c.client_hash_id = d.client_hash_id
    WHERE d.month IN ('2026-01','2026-02','2026-03','2026-04')
      AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{LABEL_END}'
      AND d.report_date >= CAST(c.gsc_data_start AS DATE)
),
agg AS (
    SELECT client_hash_id, content_hash_id,
      SUM(gsc_impressions) * 1.0 / COUNT(report_date)                         AS f01,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)                 AS f02,
      SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)           AS f05,
      SUM(CASE WHEN ga4_ok THEN ga4_engaged_sessions END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_ok THEN ga4_sessions END), 0)              AS f06,
      SUM(CASE WHEN ga4_ok THEN ga4_total_engagement_sec END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_ok THEN ga4_sessions END), 0)              AS f07,
      SUM(CASE WHEN ga4_ok THEN ga4_pageviews END) * 1.0
        / NULLIF(COUNT(CASE WHEN ga4_ok THEN report_date END), 0)             AS f08,
      SUM(CASE WHEN ga4_ok THEN ga4_users END) * 1.0
        / NULLIF(COUNT(CASE WHEN ga4_ok THEN report_date END), 0)             AS f09
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2
),
halves AS (
    SELECT client_hash_id, content_hash_id,
      (report_date > DATE '{MID}') AS is_h2,
      SUM(gsc_impressions) * 1.0 / COUNT(report_date)                AS avg_impr,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)        AS ctr,
      SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)  AS pos
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2, 3
    HAVING SUM(gsc_impressions) > 0
),
trend AS (
    SELECT client_hash_id, content_hash_id,
      (MAX(CASE WHEN is_h2 THEN avg_impr END) - MAX(CASE WHEN NOT is_h2 THEN avg_impr END))
        / NULLIF(MAX(CASE WHEN NOT is_h2 THEN avg_impr END), 0)       AS f03,
      (MAX(CASE WHEN is_h2 THEN ctr END) - MAX(CASE WHEN NOT is_h2 THEN ctr END))
        / NULLIF(MAX(CASE WHEN NOT is_h2 THEN ctr END), 0)            AS f04,
      MAX(CASE WHEN is_h2 THEN pos END) - MAX(CASE WHEN NOT is_h2 THEN pos END) AS f14
    FROM halves
    GROUP BY 1, 2
),
dim AS (
    SELECT content_hash_id,
      DATE_DIFF('day',
        CAST(CASE WHEN content_updated_date IS NOT NULL
                   AND content_updated_date <= DATE '{FEAT_END}'
                  THEN content_updated_date ELSE content_created_date END AS DATE),
        DATE '{FEAT_END}')                                                   AS f10,
      CASE WHEN keyword_hash_id IS NULL OR keyword_hash_id = '' THEN NULL
           ELSE COUNT(*) OVER (PARTITION BY client_hash_id, keyword_hash_id) - 1
      END                                                                    AS f11,
      CASE WHEN main_intent IS NULL THEN NULL
           ELSE DENSE_RANK() OVER (ORDER BY main_intent) - 1 END             AS f12,
      backlinks                                                              AS f13
    FROM read_parquet('{SRC_CONTENT}')
),
lab AS (
    SELECT client_hash_id, content_hash_id,
      SUM(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
               THEN gsc_impressions END) * 1.0
        / NULLIF(COUNT(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
               THEN report_date END), 0)                                     AS avg_base,
      COALESCE(SUM(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
               THEN gsc_impressions END) * 1.0
        / NULLIF(COUNT(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
               THEN report_date END), 0), 0)                                 AS avg_next
    FROM daily
    GROUP BY 1, 2
)
SELECT s.content_hash_id AS content_id,
       s.client_hash_id  AS client_id,
       a.f01 AS avg_daily_impressions,
       a.f02 AS ctr,
       t.f03 AS impressions_trend,
       t.f04 AS ctr_trend,
       a.f05 AS weighted_avg_position,
       a.f06 AS engagement_rate,
       a.f07 AS engagement_time_per_session,
       a.f08 AS avg_daily_pageviews,
       a.f09 AS avg_daily_users,
       m.f10 AS page_age_days,
       m.f11 AS same_keyword_pages,
       m.f12 AS main_intent_code,
       m.f13 AS backlinks,
       t.f14 AS position_trend,
       CASE WHEN l.avg_next < 0.8 * l.avg_base THEN TRUE
            ELSE FALSE END AS is_declining
FROM content_slice s
LEFT JOIN agg   a ON a.client_hash_id = s.client_hash_id AND a.content_hash_id = s.content_hash_id
LEFT JOIN trend t ON t.client_hash_id = s.client_hash_id AND t.content_hash_id = s.content_hash_id
LEFT JOIN lab   l ON l.client_hash_id = s.client_hash_id AND l.content_hash_id = s.content_hash_id
LEFT JOIN dim   m ON m.content_hash_id = s.content_hash_id
""")

df = con.execute("SELECT * FROM dataset").df()
print(df.shape)
df

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

client grouped split

In [ ]:
import numpy as np

TEST_PAGES       = 0.20
PAGES_TOL        = 0.02
MIN_SHARE        = 0.25
MIN_TEST_CLIENTS = 5
MAX_TRIES        = 2000


def split_by_client(df, label_col="is_declining", group_col="client_id",
                    test_pages=TEST_PAGES, tol=PAGES_TOL,
                    min_share=MIN_SHARE, max_tries=MAX_TRIES,
                    min_test_clients=MIN_TEST_CLIENTS, start_seed=0):
    clients = np.sort(df[group_col].unique())
    classes = df[label_col].unique()
    sizes = df[group_col].value_counts()

    for seed in range(start_seed, max_tries):
        rng = np.random.default_rng(seed)
        shuffled = rng.permutation(clients)


        target = test_pages * len(df)
        test_clients, total = [], 0
        for c in shuffled:
            if total >= target:
                break
            test_clients.append(c)
            total += sizes[c]

        is_test = df[group_col].isin(test_clients)
        train, test = df[~is_test], df[is_test]

        if abs(len(test) / len(df) - test_pages) > tol:
            continue
        if test[group_col].nunique() < min_test_clients:
            continue

        tr_share = train[label_col].value_counts(normalize=True)
        te_share = test[label_col].value_counts(normalize=True)

        if all(tr_share.get(c, 0) >= min_share and te_share.get(c, 0) >= min_share
               for c in classes):
            return train.copy(), test.copy(), seed

    raise RuntimeError(f"No valid split in {max_tries} seeds")


train_df, test_df, seed = split_by_client(df, start_seed=0)

print("seed:", seed)
print("train:", len(train_df), "pages |", train_df["client_id"].nunique(), "clients")
print("test: ", len(test_df), "pages |", test_df["client_id"].nunique(), "clients")
print("\ntrain label share:\n", train_df["is_declining"].value_counts(normalize=True).round(3))
print("\ntest label share:\n", test_df["is_declining"].value_counts(normalize=True).round(3))

assert set(train_df["client_id"]).isdisjoint(test_df["client_id"])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

FEATURES = [
    "avg_daily_impressions", "ctr", "impressions_trend", "ctr_trend",
    "weighted_avg_position", "engagement_rate", "engagement_time_per_session",
    "avg_daily_pageviews", "avg_daily_users", "page_age_days",
    "same_keyword_pages", "main_intent_code", "backlinks", "position_trend",
]



X_train, y_train = train_df[FEATURES], train_df["is_declining"]
X_test,  y_test_grouped  = test_df[FEATURES],  test_df["is_declining"]

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=30,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

pred_grouped = rf.predict(X_test)
prob_grouped = rf.predict_proba(X_test)[:, 1]
print("test accuracy:", round(accuracy_score(y_test_grouped, pred_grouped), 4))
from sklearn.metrics import f1_score
print("macro F1:", round(f1_score(y_test_grouped, pred_grouped, average="macro"), 4))

random split

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

train_df, test_df = train_test_split(
    df,
    test_size=len(test_df),
    stratify=df["is_declining"],
    random_state=SEED,
)
train_df, test_df = train_df.copy(), test_df.copy()

print("seed:", SEED)
print("train:", len(train_df), "pages |", train_df["client_id"].nunique(), "clients")
print("test: ", len(test_df), "pages |", test_df["client_id"].nunique(), "clients")
print("\ntrain label share:\n", train_df["is_declining"].value_counts(normalize=True).round(3))
print("\ntest label share:\n", test_df["is_declining"].value_counts(normalize=True).round(3))


shared = set(train_df["client_id"]) & set(test_df["client_id"])
print("\nclients in both train and test:", len(shared))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score



X_train, y_train = train_df[FEATURES], train_df["is_declining"]
X_test,  y_test  = test_df[FEATURES],  test_df["is_declining"]

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=30,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
prob = rf.predict_proba(X_test)[:, 1]
print("test accuracy:", round(accuracy_score(y_test, pred), 4))
from sklearn.metrics import f1_score
print("macro F1:", round(f1_score(y_test, pred, average="macro"), 4))

comparing the split methods

In [ ]:
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score




def evaluate(y, p, ks=(100, 200, 500, 1000), thr=0.5):
    y = pd.Series(y).astype(int).values
    order = (-p).argsort()
    y_sorted = y[order]
    out = {f"Precision@{k}": y_sorted[:k].mean() for k in ks}
    out["Average precision"] = average_precision_score(y, p)
    out["ROC AUC"] = roc_auc_score(y, p)
    out["F1"] = f1_score(y, (p >= thr).astype(int))
    return out

res = pd.DataFrame({
    "Random split": evaluate(y_test, prob),
    "client grouped split": evaluate(y_test_grouped, prob_grouped),
}).T.round(3)

res

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
leak_df = con.execute(f"""
SELECT d.client_hash_id  AS client_id,
       d.content_hash_id AS content_id,
       SUM(d.gsc_impressions) * 1.0 / COUNT(d.report_date)          AS avg_impr_4m,
       SUM(d.gsc_clicks) * 1.0 / NULLIF(SUM(d.gsc_impressions), 0)  AS ctr_4m
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN content_slice s
  ON s.client_hash_id = d.client_hash_id AND s.content_hash_id = d.content_hash_id
JOIN read_parquet('{SRC_CLIENTS}') c ON c.client_hash_id = d.client_hash_id
WHERE d.month IN ('2026-01','2026-02','2026-03','2026-04')
  AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{LABEL_END}'
  AND d.report_date >= CAST(c.gsc_data_start AS DATE)
GROUP BY 1, 2
""").df()

print(leak_df.shape)
leak_df.head()

In [ ]:
from sklearn.model_selection import train_test_split

leaked_df = df.merge(leak_df, on=["client_id", "content_id"], how="left")

SEED = 42

train_leak, test_leak = train_test_split(
    leaked_df,
    test_size=16976,
    stratify=leaked_df["is_declining"],
    random_state=SEED,
)
train_leak, test_leak = train_leak.copy(), test_leak.copy()

print("train:", len(train_leak), "pages")
print("test: ", len(test_leak), "pages")
print("\ntest label share:\n", test_leak["is_declining"].value_counts(normalize=True).round(3))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

FEATURES_LEAK = FEATURES + ["avg_impr_4m", "ctr_4m"]

X_train_leak, y_train_leak = train_leak[FEATURES_LEAK], train_leak["is_declining"]
X_test_leak,  y_test_leak  = test_leak[FEATURES_LEAK],  test_leak["is_declining"]

rf_leak = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=30,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)
rf_leak.fit(X_train_leak, y_train_leak)

pred_leak = rf_leak.predict(X_test_leak)
prob_leak = rf_leak.predict_proba(X_test_leak)[:, 1]
print("test accuracy:", round(accuracy_score(y_test_leak, pred_leak), 4))
print("macro F1:", round(f1_score(y_test_leak, pred_leak, average="macro"), 4))

In [ ]:
res = pd.DataFrame({
    "original features": evaluate(y_test, prob,ks=()),
    "leaked features": evaluate(y_test_leak, prob_leak,ks=()),
}).T.round(3)
res

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


**"A good result is Precision@100 above 75%."**

The table shows that we achieved Precision@100, Precision@200, and Precision@500 above 75%, both with a standard random split and with a client-based split.


In [ ]:
res = pd.DataFrame({
    "Random split": evaluate(y_test, prob),
    "client grouped split": evaluate(y_test_grouped, prob_grouped),
}).T.round(3)

res

The main weakness is that some client-based splits give poor results. The standard train/test split performs well, but when we change the random seed for the client split, some of the resulting splits perform noticeably worse.

In [ ]:
train_df, test_df, seed = split_by_client(df, start_seed=100)

print("seed:", seed)
print("train:", len(train_df), "pages |", train_df["client_id"].nunique(), "clients")
print("test: ", len(test_df), "pages |", test_df["client_id"].nunique(), "clients")
print("\ntrain label share:\n", train_df["is_declining"].value_counts(normalize=True).round(3))
print("\ntest label share:\n", test_df["is_declining"].value_counts(normalize=True).round(3))

assert set(train_df["client_id"]).isdisjoint(test_df["client_id"])

In [ ]:
FEATURES = [
    "avg_daily_impressions", "ctr", "impressions_trend", "ctr_trend",
    "weighted_avg_position", "engagement_rate", "engagement_time_per_session",
    "avg_daily_pageviews", "avg_daily_users", "page_age_days",
    "same_keyword_pages", "main_intent_code", "backlinks", "position_trend",
]



X_train, y_train = train_df[FEATURES], train_df["is_declining"]
X_test,  y_test_grouped  = test_df[FEATURES],  test_df["is_declining"]

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=30,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

pred_grouped = rf.predict(X_test)
prob_grouped = rf.predict_proba(X_test)[:, 1]

evaluate(y_test, prob_grouped)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.